In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Run once in your environment:
!pip install sentence-transformers torch openpyxl

In [ ]:
import itertools
import numpy as np
import pandas as pd
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

In [ ]:
def read_and_split_snippets(file_path):
    with open(file_path, 'r', encoding='utf-8-sig') as f:
        content = f.read()

    snippets = content.split('|')
    snippets = [s.strip() for s in snippets if s.strip()]
    arrays = [snippets[i:i+10] for i in range(0, 120, 10)]

    print(f"Number of arrays: {len(arrays)}")  # Should be 12
    print(f"Size of each array: {[len(arr) for arr in arrays]}")  # Should be [10, 10, ..., 10]

    return arrays

In [ ]:
def compute_similarity_results(arrays, language):
    model = SentenceTransformer("BAAI/bge-m3")

    all_texts = [s for arr in arrays for s in arr]  # 120 strings total
    all_embeddings = model.encode(all_texts, convert_to_tensor=True)
    embeddings_arrays = [all_embeddings[i:i+10] for i in range(0, 120, 10)]

    matrix_titles = [
        "1-ICE-", "2-ICE-", "3-ICE-", "4-ICE-",
        "1-IRN-", "2-IRN-", "3-IRN-", "4-IRN-",
        "1-ASK-", "2-ASK-", "3-ASK-", "4-ASK-",
    ]

    matrix_titles = [title + language for title in matrix_titles]

    all_rows = []

    for z in range(12):
        row = {"set": matrix_titles[z]}

        for j in range(1, 10):  # texts 2..10
            score = F.cosine_similarity(
                embeddings_arrays[z][0].unsqueeze(0),  # text 1
                embeddings_arrays[z][j].unsqueeze(0)   # text 2..10
            ).item()

            row[f"1_vs_{j+1}"] = score

        all_rows.append(row)

    results_df = pd.DataFrame(all_rows)
    results_df = results_df.round(4)

    return results_df

In [ ]:
def create_boxplot(results_df, language, model, setAxis = False, minAxis = 0.0, maxAxis = 1.0):
    score_cols = [col for col in results_df.columns if col.startswith("1_vs_")]

    labels = [
      "1-ICE-", "1-IRN-", "1-ASK-",
      "2-ICE-", "2-IRN-", "2-ASK-",
      "3-ICE-", "3-IRN-", "3-ASK-",
      "4-ICE-", "4-IRN-", "4-ASK-"
      ]

    labels = [title + language for title in labels]


    plot_df = results_df.set_index("set").loc[labels]
    box_data = [plot_df.loc[label, score_cols].values.astype(float) for label in labels]

    positions = [1, 2, 3, 5, 6, 7, 9, 10, 11, 13, 14, 15]

    box_colors = [
        "#4C78A8", "#F58518", "#54A24B",
        "#4C78A8", "#F58518", "#54A24B",
        "#4C78A8", "#F58518", "#54A24B",
        "#4C78A8", "#F58518", "#54A24B"
    ]

    fig, ax = plt.subplots(figsize=(15, 7), dpi=150)

    bp = ax.boxplot(
        box_data,
        positions=positions,
        widths=0.65,
        patch_artist=True,
        showmeans=True,
        medianprops=dict(color="#222222", linewidth=2),
        whiskerprops=dict(color="#666666", linewidth=1.2),
        capprops=dict(color="#666666", linewidth=1.2),
        meanprops=dict(marker="o", markerfacecolor="white", markeredgecolor="#222222", markersize=5),
        flierprops=dict(marker="o", markerfacecolor="#999999", markeredgecolor="#999999", markersize=3, alpha=0.5)
    )

    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
        patch.set_edgecolor("#444444")
        patch.set_linewidth(1.2)

    # Make outlier dots the same color as each box
    for flier, color in zip(bp["fliers"], box_colors):
        flier.set_marker("o")
        flier.set_markerfacecolor(color)
        flier.set_markeredgecolor(color)
        flier.set_markersize(7)
        flier.set_alpha(0.7)

    # Make mean dots the same color as each box
    for mean, color in zip(bp["means"], box_colors):
        mean.set_marker("o")
        mean.set_markersize(7)

    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=0, fontsize=11)
    ax.set_ylabel("Cosine similarity", fontsize=13, labelpad=10)

    if setAxis:
      ax.set_ylim(minAxis, maxAxis)

    for x in [4, 8, 12]:
        ax.axvline(x, color="#C7C7C7", linestyle="-", linewidth=1)

    y_top = ax.get_ylim()[1]
    ax.text(2, y_top, "1", ha="center", va="bottom", fontsize=12, fontweight="bold")
    ax.text(6, y_top, "2", ha="center", va="bottom", fontsize=12, fontweight="bold")
    ax.text(10, y_top, "3", ha="center", va="bottom", fontsize=12, fontweight="bold")
    ax.text(14, y_top, "4", ha="center", va="bottom", fontsize=12, fontweight="bold")

    ax.yaxis.grid(True, linestyle="--", linewidth=0.7, alpha=0.35)
    ax.xaxis.grid(False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"Boxplot_{language}_{model}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
def export_results(results_df, language, model):
    output_path = f"Similarity_{language}_{model}.xlsx"

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        results_df.to_excel(writer, sheet_name="SemScore", index=False)

    print(f"Results saved to {output_path}")

In [ ]:
arrays = read_and_split_snippets('Repetitions-EN-CHATGPT.txt') #change to .txt name of the file
model_name = "CHATGPT" #change to specific model
language = "EN" #change to specific language
results_df = compute_similarity_results(arrays, language)
results_df
create_boxplot(results_df, language, model_name, True, 0.7, 1.0) #change to create_boxplot(results_df, language, model_name) to get the axis automatically from the plot
export_results(results_df, language, model_name)